# EDA — Task 2: Data Preparation

This notebook prepares the datasets needed for the Task 2 optimisation model.
The main goals are:

1. **Align ZIP codes** — ensure all facilities and potential locations map to a ZIP in our population list.
2. **Fill missing coordinates** — every facility and potential location needs a lat/lon for the 0.06-mile distance constraint.

All cleaned datasets are inherited from the Task 1 EDA output (`data/task_1/`).


## 1. Imports

In [73]:
import pandas as pd
import numpy as np

## 2. Load Data

We load the four cleaned datasets produced by Task 1 EDA, plus the raw potential locations file
which has not been cleaned yet.


In [74]:
# Load cleaned datasets from Task 1 output
avg_income_nyc            = pd.read_csv('../data/task_1/avg_income_nyc.csv')
child_care_regulated_nyc  = pd.read_csv('../data/task_1/child_care_regulated_nyc.csv')
employment_rate_nyc       = pd.read_csv('../data/task_1/employment_rate_nyc.csv')
population_nyc            = pd.read_csv('../data/task_1/population_nyc.csv')
potential_locations_nyc   = pd.read_csv('../data/raw_data/potential_locations_nyc.csv', index_col=0)

## 3. Data Preview

Quick look at each dataset to confirm structure and column names.


In [75]:
avg_income_nyc.head()

,zipcode,average income
0,10001,102878.033603
1,10002,59604.041165
2,10003,114273.049645
3,10004,132004.310345
4,10005,121437.713311


In [76]:
child_care_regulated_nyc.head()

,facility_id,program_type,facility_status,facility_name,city,zipcode,school_district_name,infant_capacity,toddler_capacity,preschool_capacity,school_age_capacity,children_capacity,total_capacity,latitude,longitude
0,51349,FDC,Registration,"Valerio, Gladys",Bronx,10474,Bronx 8,0,0,0,0,6,6,40.818399,-73.888816
1,73544,SACC,Registration,YMCA OF GREATER NY,New York,10017,Manhattan 2,0,0,0,60,0,60,40.753216,-73.970794
2,108407,GFDC,License,"Almond Tree Group Family Day Care, LLC",Brooklyn,11225,Brooklyn 17,0,0,0,4,12,16,NaN,NaN
3,111953,GFDC,License,"Williams, Petal",Brooklyn,11226,Brooklyn 22,0,0,0,4,12,16,NaN,NaN
4,126425,GFDC,License,"Hernandez, Lidia",Bronx,10465,Bronx 8,0,0,0,0,12,12,40.841228,-73.823572


In [77]:
employment_rate_nyc.head()

,zipcode,employment rate
0,10001,0.595097
1,10002,0.520662
2,10003,0.497244
3,10004,0.506661
4,10005,0.665833


In [78]:
population_nyc.head()

,zipcode,Total,-5,6-12,13-14,15-19,20-24,25-29,30-34,35-39,40-44,45-49,50-54,55-59,60-64,65-69,70-74,75-79,80-84,85+
0,10001,27004,744,1255,471,1035,2296,3806,3588,2524,1702,1903,1704,1225,1323,933,815,616,488,576
1,10002,76518,2142,4645,1599,2652,4528,6988,6278,5157,4962,4822,4410,6106,4548,4815,4748,2531,2793,2794
2,10003,53877,1440,1510,477,7013,6344,7100,6427,3221,2907,1988,2698,2350,2274,2793,1854,1646,779,1056
3,10004,4579,433,262,81,108,109,601,724,490,241,313,549,279,199,173,2,15,0,0
4,10005,8801,484,318,115,53,989,2604,1144,945,685,351,652,218,85,92,66,0,0,0


In [79]:
potential_locations_nyc.head()

,zipcode,latitude,longitude
0,10001,40.741893,-74.000140
1,10001,40.752007,-74.005436
2,10001,40.750545,-73.997147
3,10001,40.744080,-74.001932
4,10001,40.748690,-73.999341


## 4. ZIP Code Alignment — Potential Locations

`potential_locations_nyc` contains ZIP codes for every candidate building site.
We need every ZIP to exist in our population list so the optimisation model can
assign new facilities to the correct desert zip zone.

**Step 1:** Count how many ZIPs are misaligned.


In [80]:
# Unique zip codes in potential_locations_nyc
pot_zips = set(potential_locations_nyc['zipcode'].dropna().astype(int))
pop_zips = set(population_nyc['zipcode'].dropna().astype(int))

only_in_potential  = pot_zips - pop_zips
only_in_population = pop_zips - pot_zips

print(f"Unique ZIP codes in potential_locations_nyc : {len(pot_zips)}")
print(f"Unique ZIP codes in population_nyc          : {len(pop_zips)}")
print(f"ZIP codes in both                           : {len(pot_zips & pop_zips)}")
print()
print(f"Only in potential_locations_nyc : {sorted(only_in_potential) or 'None'}")
print(f"Only in population_nyc          : {sorted(only_in_population) or 'None'}")

Unique ZIP codes in potential_locations_nyc : 311
Unique ZIP codes in population_nyc          : 180
ZIP codes in both                           : 180

Only in potential_locations_nyc : [10008, 10020, 10041, 10043, 10045, 10055, 10060, 10080, 10081, 10087, 10090, 10101, 10102, 10103, 10104, 10105, 10106, 10107, 10108, 10109, 10110, 10111, 10112, 10113, 10114, 10115, 10116, 10117, 10118, 10119, 10120, 10121, 10122, 10123, 10124, 10125, 10126, 10129, 10130, 10131, 10132, 10133, 10138, 10150, 10151, 10152, 10153, 10154, 10155, 10156, 10157, 10158, 10159, 10160, 10161, 10163, 10164, 10165, 10166, 10167, 10168, 10169, 10170, 10171, 10172, 10173, 10174, 10175, 10176, 10177, 10178, 10179, 10185, 10199, 10203, 10211, 10212, 10213, 10242, 10249, 10256, 10258, 10259, 10260, 10261, 10265, 10268, 10269, 10270, 10271, 10272, 10273, 10274, 10275, 10276, 10277, 10278, 10279, 10281, 10285, 10286, 10311, 10313, 11120, 11202, 11241, 11242, 11243, 11245, 11247, 11251, 11252, 11256, 11351, 11352, 11359, 11

### 4.1 Manual Remapping — Known Commercial / PO-Box ZIPs

Many out-of-population ZIPs are well-known NYC commercial codes (e.g. Rockefeller Center,
Grand Central corridor, Penn Station cluster). We manually map these to the residential
ZIP that physically contains them.


In [81]:
# Reuse the building→area zip mapping from Task 1 EDA
building_zip_mapping = {
    # Rockefeller Center cluster → Midtown West
    10020: 10036, 10103: 10036, 10110: 10036, 10111: 10036, 10112: 10036,
    # Upper West Side
    10115: 10025,
    # Penn Station / MSG cluster → Chelsea/Hudson Yards
    10119: 10001, 10173: 10001, 10199: 10001,
    # Grand Central / Park Ave cluster → Murray Hill
    10154: 10017, 10165: 10017, 10166: 10017, 10167: 10017,
    10168: 10017, 10169: 10017, 10174: 10017, 10177: 10017,
    # Midtown East / 3rd Ave cluster → Midtown East
    10152: 10022, 10153: 10022, 10170: 10022, 10171: 10022, 10172: 10022,
    # Financial District
    10271: 10005, 10278: 10007,
    # Staten Island
    10311: 10314,
    # Queens
    11359: 11360, 11371: 11369, 11424: 11432, 11451: 11432,
}

can_remap   = {z: building_zip_mapping[z] for z in only_in_potential if z in building_zip_mapping}
cannot_remap = only_in_potential - set(can_remap.keys())

print(f"Zips only in potential_locations : {len(only_in_potential)}")
print(f"  → Can be remapped via building_zip_mapping : {len(can_remap)}")
print(f"  → Cannot be remapped (no mapping found)   : {len(cannot_remap)}")
print()
if can_remap:
    print("Remappable:")
    for z, target in sorted(can_remap.items()):
        print(f"  {z} → {target}")
if cannot_remap:
    print("\nNo mapping found for:")
    print(f"  {sorted(cannot_remap)}")

Zips only in potential_locations : 131
  → Can be remapped via building_zip_mapping : 29
  → Cannot be remapped (no mapping found)   : 102

Remappable:
  10020 → 10036
  10103 → 10036
  10110 → 10036
  10111 → 10036
  10112 → 10036
  10115 → 10025
  10119 → 10001
  10152 → 10022
  10153 → 10022
  10154 → 10017
  10165 → 10017
  10166 → 10017
  10167 → 10017
  10168 → 10017
  10169 → 10017
  10170 → 10022
  10171 → 10022
  10172 → 10022
  10173 → 10001
  10174 → 10017
  10177 → 10017
  10199 → 10001
  10271 → 10005
  10278 → 10007
  10311 → 10314
  11359 → 11360
  11371 → 11369
  11424 → 11432
  11451 → 11432

No mapping found for:
  [10008, 10041, 10043, 10045, 10055, 10060, 10080, 10081, 10087, 10090, 10101, 10102, 10104, 10105, 10106, 10107, 10108, 10109, 10113, 10114, 10116, 10117, 10118, 10120, 10121, 10122, 10123, 10124, 10125, 10126, 10129, 10130, 10131, 10132, 10133, 10138, 10150, 10151, 10155, 10156, 10157, 10158, 10159, 10160, 10161, 10163, 10164, 10175, 10176, 10178, 10179, 1

### 4.2 Geocode Remaining Unmapped ZIPs

For the ZIPs that have no manual mapping, we use **pgeocode** to retrieve their centroid
coordinates, which we will then use to find the nearest population ZIP by great-circle distance.


In [82]:
import pgeocode

nomi = pgeocode.Nominatim('us')

geo = nomi.query_postal_code([str(z) for z in sorted(cannot_remap)])

geo_df = pd.DataFrame({
    'zipcode':    sorted(cannot_remap),
    'place_name': geo['place_name'].values,
    'county':     geo['county_name'].values,
    'lat':        geo['latitude'].values,
    'lon':        geo['longitude'].values,
})

print(f"Total unmapped zips: {len(geo_df)}")
print()
print(geo_df.to_string(index=False))

Total unmapped zips: 102

 zipcode       place_name   county     lat      lon
   10008         New York New York 40.7143 -74.0060
   10041         New York New York 40.7038 -74.0098
   10043         New York New York 40.7143 -74.0060
   10045         New York New York 40.7086 -74.0087
   10055         New York New York 40.7808 -73.9772
   10060         New York New York 40.7808 -73.9772
   10080         New York New York 40.7143 -74.0060
   10081         New York New York 40.7143 -74.0060
   10087         New York New York 40.7808 -73.9772
   10090         New York New York 40.7808 -73.9772
   10101         New York New York 40.7808 -73.9772
   10102         New York New York 40.7808 -73.9772
   10104         New York New York 40.7609 -73.9799
   10105         New York New York 40.7628 -73.9785
   10106         New York New York 40.7652 -73.9804
   10107         New York New York 40.7664 -73.9827
   10108         New York New York 40.7808 -73.9772
   10109         New York New York 40.

### 4.3 Nearest-ZIP Matching via Haversine Distance

We fetch the centroid of every population ZIP and, for each unmapped ZIP,
find the closest population ZIP using the Haversine formula (great-circle distance in miles).
This gives us an automatic, geographically sensible fallback mapping.


In [83]:
# Get lat/lon for all population zip codes
pop_zip_list = sorted(pop_zips)
pop_geo      = nomi.query_postal_code([str(z) for z in pop_zip_list])
pop_geo_df   = pd.DataFrame({
    'zipcode': pop_zip_list,
    'lat':     pop_geo['latitude'].values,
    'lon':     pop_geo['longitude'].values,
}).dropna(subset=['lat', 'lon'])

def haversine(lat1, lon1, lat2, lon2):
    R = 3958.8  # Earth radius in miles
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi  = np.radians(lat2 - lat1)
    dlam  = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlam/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# For each unmapped zip, find the nearest pop_zip by great-circle distance
nearest_mapping = {}
for _, row in geo_df.iterrows():
    if pd.isna(row['lat']) or pd.isna(row['lon']):
        print(f"  ⚠ {int(row['zipcode'])} has no coordinates — skipping")
        continue
    dists = haversine(row['lat'], row['lon'],
                      pop_geo_df['lat'].values, pop_geo_df['lon'].values)
    best_idx = np.argmin(dists)
    nearest_zip  = int(pop_geo_df.iloc[best_idx]['zipcode'])
    nearest_dist = dists[best_idx]
    nearest_mapping[int(row['zipcode'])] = nearest_zip
    print(f"  {int(row['zipcode'])} ({row['place_name']}) → {nearest_zip}  ({nearest_dist:.2f} mi)")

print(f"Nearest-zip mapping built for {len(nearest_mapping)} unmapped zips.")


  10008 (New York) → 10004  (0.00 mi)
  10041 (New York) → 10005  (0.15 mi)
  10043 (New York) → 10004  (0.00 mi)
  10045 (New York) → 10005  (0.21 mi)
  10055 (New York) → 10024  (0.39 mi)
  10060 (New York) → 10024  (0.39 mi)
  10080 (New York) → 10004  (0.00 mi)
  10081 (New York) → 10004  (0.00 mi)
  10087 (New York) → 10024  (0.39 mi)
  10090 (New York) → 10024  (0.39 mi)
  10101 (New York) → 10024  (0.39 mi)
  10102 (New York) → 10024  (0.39 mi)
  10104 (New York) → 10019  (0.42 mi)
  10105 (New York) → 10019  (0.41 mi)
  10106 (New York) → 10019  (0.28 mi)
  10107 (New York) → 10019  (0.19 mi)
  10108 (New York) → 10024  (0.39 mi)
  10109 (New York) → 10024  (0.39 mi)
  10113 (New York) → 10024  (0.39 mi)
  10114 (New York) → 10024  (0.39 mi)
  10116 (New York) → 10024  (0.39 mi)
  10117 (New York) → 10024  (0.39 mi)
  10118 (New York) → 10018  (0.50 mi)
  10120 (New York) → 10018  (0.33 mi)
  10121 (New York) → 10001  (0.26 mi)
  10122 (New York) → 10018  (0.20 mi)
  10123 (New

### 4.4 Apply Full Remapping to `potential_locations_nyc`

We merge the manual mapping and the nearest-ZIP mapping into a single lookup dict
(`full_zip_remap`) and apply it to the dataset in one pass.


In [84]:
# Merge nearest_mapping into the full zip remapping dict and apply to potential_locations_nyc
full_zip_remap = {**building_zip_mapping, **nearest_mapping}

potential_locations_nyc['zipcode'] = (
    potential_locations_nyc['zipcode']
    .astype(int)
    .map(lambda z: full_zip_remap.get(z, z))
)

# Verify no out-of-population zips remain
remaining = set(potential_locations_nyc['zipcode'].astype(int)) - pop_zips
print(f"Zips still outside population list after remapping: {sorted(remaining) or 'None'}")
print(f"potential_locations_nyc shape: {potential_locations_nyc.shape}")


Zips still outside population list after remapping: None
potential_locations_nyc shape: (31100, 3)


### 4.5 Verify — ZIP Alignment After Remapping


In [85]:
pot_zips_remapped = set(potential_locations_nyc['zipcode'].astype(int))

only_in_potential_after  = pot_zips_remapped - pop_zips
only_in_population_after = pop_zips - pot_zips_remapped

print(f"After remapping:")
print(f"  Unique ZIPs in potential_locations_nyc : {len(pot_zips_remapped)}")
print(f"  Unique ZIPs in population_nyc          : {len(pop_zips)}")
print(f"  ZIPs in both                           : {len(pot_zips_remapped & pop_zips)}")
print(f"  Only in potential (unresolved)         : {sorted(only_in_potential_after) or 'None'}")
print(f"  Only in population (no locations)      : {sorted(only_in_population_after) or 'None'}")


After remapping:
  Unique ZIPs in potential_locations_nyc : 180
  Unique ZIPs in population_nyc          : 180
  ZIPs in both                           : 180
  Only in potential (unresolved)         : None
  Only in population (no locations)      : None


## 5. Missing Coordinates — Child Care Facilities

Task 2 imposes a **minimum distance of 0.06 miles** between any two facilities.
This means every facility must have valid lat/lon coordinates.

**Step 1:** Audit how many rows are missing coordinates.


In [86]:
# Check for missing lat/lon in both spatial datasets
for name, df in [('potential_locations_nyc', potential_locations_nyc),
                 ('child_care_regulated_nyc', child_care_regulated_nyc)]:
    n_total    = len(df)
    n_lat_nan  = df['latitude'].isna().sum()
    n_lon_nan  = df['longitude'].isna().sum()
    n_both_nan = df[df['latitude'].isna() & df['longitude'].isna()].shape[0]
    print(f"{'─'*50}")
    print(f"  {name}  ({n_total} rows)")
    print(f"  latitude  NaNs : {n_lat_nan}  ({100*n_lat_nan/n_total:.1f}%)")
    print(f"  longitude NaNs : {n_lon_nan}  ({100*n_lon_nan/n_total:.1f}%)")
    print(f"  both NaN       : {n_both_nan}  ({100*n_both_nan/n_total:.1f}%)")
print(f"{'─'*50}")


──────────────────────────────────────────────────
  potential_locations_nyc  (31100 rows)
  latitude  NaNs : 0  (0.0%)
  longitude NaNs : 0  (0.0%)
  both NaN       : 0  (0.0%)
──────────────────────────────────────────────────
  child_care_regulated_nyc  (7740 rows)
  latitude  NaNs : 169  (2.2%)
  longitude NaNs : 169  (2.2%)
  both NaN       : 169  (2.2%)
──────────────────────────────────────────────────


### 5.1 Fill Missing Coordinates via Google Maps Geocoding

For each facility with missing lat/lon we:
1. Construct a search query from `facility_name` + `city` + `", NY"`.
2. Call the **Google Maps Geocoding API** to retrieve the precise location of that facility.
3. Fall back to the ZIP centroid (from `pop_geo_df`) **only if** geocoding returns no result.

This avoids assigning a coarse area centroid to a facility that may actually sit close to
(or far from) another facility, which would distort the 0.06-mile distance constraint.

In [90]:
import googlemaps
from dotenv import load_dotenv
import os

load_dotenv()
gmaps = googlemaps.Client(key=os.environ["GOOGLE_MAPS_API_KEY"])

missing_mask = child_care_regulated_nyc['latitude'].isna()
print(f"Facilities missing lat/lon before fill: {missing_mask.sum()}")

def geocode_facility(row):
    """Return (lat, lon) for a facility using Google Maps geocoding."""
    if not pd.isna(row['latitude']):
        return row['latitude'], row['longitude']

    name  = str(row.get('facility_name', '')).strip()
    city  = str(row.get('city', '')).strip()
    query = f"{name}, {city}, NY" if city else f"{name}, New York, NY"

    try:
        results = gmaps.geocode(query, region='us')
        if results:
            loc = results[0]['geometry']['location']
            return loc['lat'], loc['lng']
    except Exception as e:
        print(f"  ⚠ Geocoding failed for '{query}': {e}")

    return np.nan, np.nan

filled = child_care_regulated_nyc[missing_mask].apply(
    geocode_facility, axis=1, result_type='expand'
)
filled.columns = ['latitude', 'longitude']

child_care_regulated_nyc.loc[missing_mask, 'latitude']  = filled['latitude'].values
child_care_regulated_nyc.loc[missing_mask, 'longitude'] = filled['longitude'].values

still_missing = child_care_regulated_nyc['latitude'].isna().sum()
print(f"Facilities missing lat/lon after  fill: {still_missing}")

Facilities missing lat/lon before fill: 169
Facilities missing lat/lon after  fill: 0


## 6. ZIP Code Alignment — Child Care Facilities

Final check: verify that the child care facility ZIPs are aligned with the population list.
ZIPs that appear only in the population (no facilities) are expected — these are the desert
zip codes with no existing provision.


In [91]:
cc_zips = set(child_care_regulated_nyc['zipcode'].dropna().astype(int))

only_in_cc  = cc_zips - pop_zips
only_in_pop = pop_zips - cc_zips

print(f"Unique ZIPs in child_care_regulated_nyc : {len(cc_zips)}")
print(f"Unique ZIPs in population_nyc           : {len(pop_zips)}")
print(f"ZIPs in both                            : {len(cc_zips & pop_zips)}")
print(f"Only in child_care (unresolved)         : {sorted(only_in_cc) or 'None'}")
print(f"Only in population (no facilities)      : {sorted(only_in_pop) or 'None'}")


Unique ZIPs in child_care_regulated_nyc : 176
Unique ZIPs in population_nyc           : 180
ZIPs in both                            : 176
Only in child_care (unresolved)         : None
Only in population (no facilities)      : [10018, 10162, 11005, 11697]


## 7. Save Cleaned Datasets

Export the cleaned and ZIP-aligned datasets to `data/task_2/` for use by the Task 2 optimisation model.


In [92]:
import os

os.makedirs('../data/task_2', exist_ok=True)

task2_saves = {
    'potential_locations_nyc.csv':  potential_locations_nyc,
    'child_care_regulated_nyc.csv': child_care_regulated_nyc,
}

for filename, df in task2_saves.items():
    df.to_csv(os.path.join('../data/task_2', filename), index=False)
    print(f"Saved {filename}  ({len(df)} rows)")


Saved potential_locations_nyc.csv  (31100 rows)
Saved child_care_regulated_nyc.csv  (7740 rows)
